# DASH-Q: Robust Ultra Low-Bit Post-Training Quantization

This notebook demonstrates the **DASH-Q** algorithm based on the paper *Robust Ultra Low-Bit Post-Training Quantization via Stable Diagonal Curvature Estimate*.

DASH-Q utilizes a diagonal Hessian approximation and iterative weighted least squares to perform robust 2-bit or 3-bit quantization while keeping the inference engine compatible with standard integer affine formats.

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Ensure we can import our source
sys.path.append(os.path.abspath('src/'))

from dashq.dash_q import dash_q_group, quantize_layer

# Formatting for plots
sns.set_theme(style="whitegrid")

## 1. Setup a Dummy Layer and Calibration Data

We start by initializing a dummy weight matrix $W$ of a linear layer and generating some synthetic calibration data $X$ to compute the Hessian.

In [ ]:
torch.manual_seed(42)

in_features = 256
out_features = 128
group_size = 64
bits = 3

# Synthetic normal distributed weights
W = torch.randn(out_features, in_features) * 0.1

# Synthetic calibration data (e.g. 128 tokens of hidden states)
N_calibration = 128
X = torch.randn(N_calibration, in_features)

# Compute diagonal Hessian (H_jj = sum_k x_{kj}^2)
H_diag = (X ** 2).sum(dim=0)

print(f"Weight shape: {W.shape}")
print(f"Hessian diagonal shape: {H_diag.shape}")

## 2. Walkthrough: Single Group Quantization

Let's isolate one group of weights and observe how DASH-Q optimizes the quantization parameters `s` (scale) and `z` (zero-point) via Coordinate Descent.

In [ ]:
W_group = W[:, :group_size]
H_group = H_diag[:group_size]

q_max = (2 ** bits) - 1
H_group_reshaped = H_group.view(1, -1)
H_sum = H_group_reshaped.sum()

# Initial Standard Affine Quantization (Min-Max)
w_min = W_group.min(dim=1, keepdim=True).values
w_max = W_group.max(dim=1, keepdim=True).values
s_init = (w_max - w_min) / q_max
z_init = -w_min / s_init

Q_init = torch.clamp(torch.round((W_group + z_init) / s_init), 0, q_max)
W_hat_init = s_init * Q_init - z_init

# Apply DASH-Q Optimization
Q_dash, s_dash, z_dash = dash_q_group(W_group, H_group, b=bits, T=9, lambda_reg=1e-2)
W_hat_dash = s_dash * Q_dash - z_dash

mse_init = torch.nn.functional.mse_loss(W_hat_init, W_group).item()
mse_dash = torch.nn.functional.mse_loss(W_hat_dash, W_group).item()

print(f"Initial Min-Max MSE: {mse_init:.6f}")
print(f"DASH-Q MSE:          {mse_dash:.6f}")
print(f"Improvement:         {(mse_init - mse_dash) / mse_init * 100:.2f}%")

## 3. Visualizing Weight Reconstruction

Following Figure 3 in the paper, we visualize how original weights are mapped to quantized outputs. A tighter alignment to the $y = x$ line indicates lower error.

The points colored by Hessian importance show how DASH-Q prioritizes reconstructing salient features with higher weights.

In [ ]:
plt.figure(figsize=(12, 5))

# Plot for Min-Max
plt.subplot(1, 2, 1)
plt.scatter(W_group.flatten(), W_hat_init.flatten(), alpha=0.5, c=H_group.repeat(out_features), cmap='coolwarm')
plt.plot([-0.3, 0.3], [-0.3, 0.3], 'k--', alpha=0.8)
plt.title('Min-Max Uniform Quantization')
plt.xlabel('Original Weight')
plt.ylabel('Reconstructed Weight')

# Plot for DASH-Q
plt.subplot(1, 2, 2)
plt.scatter(W_group.flatten(), W_hat_dash.flatten(), alpha=0.5, c=H_group.repeat(out_features), cmap='coolwarm')
plt.plot([-0.3, 0.3], [-0.3, 0.3], 'k--', alpha=0.8)
plt.title('DASH-Q Quantization')
plt.xlabel('Original Weight')

plt.tight_layout()
plt.show()

## 4. Full Layer Quantization

Finally, we use the `quantize_layer` wrapper to apply DASH-Q across the entire weight matrix, returning the discrete weights `Q`, scales `S`, and zero-points `Z` ready for inference engines.

In [ ]:
Q_full, S_full, Z_full = quantize_layer(W, X, b=bits, group_size=group_size, T=9)

print("Full layer quantization complete.")
print(f"Q_full shape: {Q_full.shape} (integers 0 to {q_max})")
print(f"S_full shape: {S_full.shape} (fp16/fp32 scales)")
print(f"Z_full shape: {Z_full.shape} (fp16/fp32 zero-points)")